# Phương án 7 (Triangle Velocities Synergy - TVS) — Colab Runner

Chạy Phương án 7 (TVS - Tam giác hoá vận tốc, xây trên Phương án 6 — Learnable Coarse Anchor) trên
[DiffMM-TVS](https://github.com/thyelmot/DiffMM_7.git) (bản fork của [HKUDS/DiffMM](https://github.com/HKUDS/DiffMM)).
Xem thiết kế đầy đủ ở
[`Phuong_An_7_TVS_KeHoachChiTiet.md`](../Phuong_An_7_TVS_KeHoachChiTiet.md)
và kết quả kiểm chứng trước khi tới notebook này ở
[`verify_cpu_tvs.py`](verify_cpu_tvs.py).

**Cách dùng:** chạy lần lượt từng cell từ trên xuống. Cell 1 là nơi duy nhất cần chỉnh mỗi lần dùng
(link GitHub + Google Drive + dataset + epoch + `VELOCITY_MODE`). Nhớ bật GPU:
`Runtime > Change runtime type > Hardware accelerator > GPU`.


## 1. Cell cấu hình đầu vào

Đây là **cell duy nhất người dùng cuối cần chỉnh sửa** mỗi lần chạy. Khi điền template, thêm mọi
hyperparameter riêng của phương án mới vào đúng đây (theo mẫu `SIGMA_MIN` đã comment sẵn bên dưới).
`DATASET_NAME` dùng để đặt tên và ghi chú trong file PDF kết quả ở Cell 7 — nếu thuật toán không có
khái niệm nhiều dataset, cứ để 1 giá trị cố định (ví dụ tên dataset duy nhất mà repo hỗ trợ).

In [ ]:
# ============================================================
# CELL 1 — CẤU HÌNH ĐẦU VÀO (chỗ DUY NHẤT người dùng cần chỉnh sửa)
# ============================================================

GITHUB_REPO_URL = "https://github.com/thyelmot/DiffMM_7.git"  # repo GitHub riêng đã push code Phương án 7
GDRIVE_LINK = "https://drive.google.com/drive/folders/1UdSihFXvm5frxb3nAxJ_uMaoZrFssqqD?usp=sharing"  # link thư mục Google Drive chứa dữ liệu
DATASET_NAME = "tiktok"  # "tiktok" | "baby" | "sports"
NUM_EPOCHS = 50         # <-- chỉnh số epoch mong muốn ở đây

# Hyperparameter riêng của Phương án 7 (xem Params.py / README của DiffMM-TVS):
SIGMA_MIN = 1e-3         # [Phương án 1, tái dùng] sigma_min cho đường OT-linear
W_CLIP = 50.0             # [Phương án 2, tái dùng] chặn trên trọng số CFM (kiểu Min-SNR)
NUM_SAMPLE_STEPS = 0      # [Phương án 3, tái dùng] 0 = tự suy round(0.6*steps)
ANCHOR_W = 2.0            # [Phương án 6, tái dùng] cường độ điểm neo
VELOCITY_MODE = 1         # [Phương án 7] 1: Bật TVS (hồi quy vận tốc), 0: Tắt TVS (hồi quy dữ liệu gốc, giống PA6)
LAMBDA_X = 1.0            # Trọng số loss quỹ đạo chính
LAMBDA_Y = 1.0            # Trọng số loss quỹ đạo phụ 1 (neo thô)
LAMBDA_Z = 1.0            # Trọng số loss quỹ đạo phụ 2 (không đổi)
PATIENCE = 5              # Kiên nhẫn dừng sớm (epochs không cải thiện); 0 để tắt

assert GITHUB_REPO_URL.strip() != "", "Hãy dán URL repo GitHub của bạn vào GITHUB_REPO_URL ở trên trước khi chạy tiếp."
assert GDRIVE_LINK.strip() != "", "Hãy dán link Google Drive vào biến GDRIVE_LINK ở trên trước khi chạy tiếp."
assert DATASET_NAME.strip() != "", "Hãy điền tên dataset vào DATASET_NAME ở trên trước khi chạy tiếp."
print(f"Cấu hình: repo={GITHUB_REPO_URL}, dataset={DATASET_NAME}, epochs={NUM_EPOCHS}, velocity_mode={VELOCITY_MODE}")


## 2. Cell setup môi trường

In [ ]:
# ============================================================
# CELL 2 — SETUP MÔI TRƯỜNG
# ============================================================
import torch

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Chưa bật GPU cho Colab. Vào Runtime > Change runtime type > Hardware accelerator > GPU, "
    "rồi Runtime > Restart session và chạy lại từ Cell 1."
)

!pip install -q gdown tabulate setproctitle

print("Đã cài đặt xong các thư viện cần thiết.")


## 3. Cell clone code

**Hạ tầng đã kiểm chứng — không cần sửa**, trừ 2 biến `REPO_DIR` và `MAIN_SCRIPT_NAME` ở đầu cell.
Cell này luôn xoá bản clone cũ rồi clone lại từ đầu mỗi lần chạy (tránh lỗi dùng nhầm code cũ khi chạy
lại trong cùng phiên Colab), và tự dò tìm entry point script trong toàn bộ cây thư mục vừa clone,
tự động điều chỉnh đường dẫn nếu repo bị lồng thêm cấp thư mục.

In [ ]:
# ============================================================
# CELL 3 — CLONE CODE TỪ REPO GITHUB CỦA BẠN
# ============================================================
import glob
import os
import shutil

REPO_DIR = "DiffMM-TVS"
MAIN_SCRIPT_NAME = "Main.py"

if os.path.isdir(REPO_DIR):
    print(f"Xoá bản clone cũ '{REPO_DIR}' để lấy code MỚI NHẤT từ GitHub...")
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 {GITHUB_REPO_URL} {REPO_DIR}

assert os.path.isdir(REPO_DIR), (
    f"Không tìm thấy thư mục '{REPO_DIR}' sau khi clone — kiểm tra lại GITHUB_REPO_URL ở Cell 1 "
    "(repo phải công khai, hoặc bạn đã đăng nhập git trên Colab nếu là repo private)."
)

main_script_candidates = glob.glob(os.path.join(REPO_DIR, "**", MAIN_SCRIPT_NAME), recursive=True)
assert main_script_candidates, (
    f"Clone thành công nhưng KHÔNG tìm thấy '{MAIN_SCRIPT_NAME}' ở đâu trong '{REPO_DIR}'.\n"
    f"Nội dung hiện có: {sorted(os.listdir(REPO_DIR))}\n"
    "Kiểm tra lại: GITHUB_REPO_URL đúng repo chưa, MAIN_SCRIPT_NAME đúng tên file chưa, và repo có "
    "đúng nội dung bản fork đã patch hay không."
)
main_script_candidates.sort(key=lambda p: p.count(os.sep))
actual_dir = os.path.dirname(main_script_candidates[0])

if actual_dir != REPO_DIR:
    print(
        f"⚠ {MAIN_SCRIPT_NAME} không nằm trực tiếp trong '{REPO_DIR}' mà nằm trong '{actual_dir}' "
        "— có thể repo bị lồng thêm 1 cấp thư mục. Tự động dùng đường dẫn này cho các bước sau."
    )
    REPO_DIR = actual_dir

print(f"\nREPO_DIR đang dùng: {REPO_DIR}")
print(sorted(os.listdir(REPO_DIR)))


## 4. Cell tải dữ liệu

**Hạ tầng đã kiểm chứng — không cần sửa**, trừ 2 biến `TARGET_DATA_DIR` và `REQUIRED_DATA_FILES` ở
đầu cell. Tự động tải từ `GDRIVE_LINK` (nhận cả link file `.zip` lẫn link thư mục), tự giải nén (kể cả
zip lồng bên trong), tự dò tìm thư mục chứa file dữ liệu đầu tiên trong `REQUIRED_DATA_FILES` bất kể
cấu trúc bên trong Drive của bạn ra sao, rồi copy đúng vào `TARGET_DATA_DIR`.

In [ ]:
# ============================================================
# CELL 4 — TẢI DỮ LIỆU TỪ GOOGLE DRIVE
# ============================================================
import glob
import shutil
import zipfile

import gdown

DATASETS_DIR = os.path.join(REPO_DIR, "Datasets")
TARGET_DATA_DIR = os.path.join(DATASETS_DIR, DATASET_NAME)
REQUIRED_DATA_FILES = ["trnMat.pkl", "tstMat.pkl", "image_feat.npy", "text_feat.npy"]
if DATASET_NAME == "tiktok":
    REQUIRED_DATA_FILES.append("audio_feat.npy")
DOWNLOAD_DIR = "gdrive_download"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(TARGET_DATA_DIR, exist_ok=True)

if "/folders/" in GDRIVE_LINK:
    print("Phát hiện link THƯ MỤC Google Drive -> tải cả thư mục...")
    gdown.download_folder(url=GDRIVE_LINK, output=DOWNLOAD_DIR, quiet=False, use_cookies=False)
else:
    print("Phát hiện link FILE Google Drive -> tải file...")
    downloaded_path = gdown.download(
        url=GDRIVE_LINK, output=os.path.join(DOWNLOAD_DIR, "gdrive_data"), quiet=False, fuzzy=True
    )
    assert downloaded_path, "Tải dữ liệu từ Google Drive thất bại — kiểm tra lại link (phải ở chế độ chia sẻ công khai/Anyone with the link)."
    if downloaded_path.lower().endswith(".zip"):
        print(f"Giải nén {downloaded_path} ...")
        with zipfile.ZipFile(downloaded_path, "r") as zf:
            zf.extractall(DOWNLOAD_DIR)

for _ in range(3):
    zip_files = glob.glob(os.path.join(DOWNLOAD_DIR, "**", "*.zip"), recursive=True)
    if not zip_files:
        break
    for zpath in zip_files:
        try:
            with zipfile.ZipFile(zpath, "r") as zf:
                zf.extractall(os.path.dirname(zpath))
            print(f"Đã giải nén: {zpath}")
            os.remove(zpath)
        except zipfile.BadZipFile:
            pass

marker_file = REQUIRED_DATA_FILES[0]
candidates = glob.glob(os.path.join(DOWNLOAD_DIR, "**", marker_file), recursive=True)
filtered_candidates = [c for c in candidates if f"/{DATASET_NAME.lower()}/" in c.lower().replace("\\", "/")]
if filtered_candidates:
    candidates = filtered_candidates
assert candidates, (
    f"Không tìm thấy '{marker_file}' trong dữ liệu tải về từ Google Drive.\n"
    "Hãy kiểm tra: (1) link đã ở chế độ chia sẻ công khai (Anyone with the link) chưa, "
    f"(2) dữ liệu có đủ các file: {REQUIRED_DATA_FILES} hay chưa."
)
src_dir = os.path.dirname(candidates[0])
print("Tìm thấy dữ liệu tại:", src_dir)

for fname in os.listdir(src_dir):
    fpath = os.path.join(src_dir, fname)
    if os.path.isfile(fpath):
        shutil.copy2(fpath, TARGET_DATA_DIR)

missing = [f for f in REQUIRED_DATA_FILES if not os.path.exists(os.path.join(TARGET_DATA_DIR, f))]
assert not missing, f"Thiếu file trong {TARGET_DATA_DIR}: {missing}. Kiểm tra lại dữ liệu trên Google Drive."

print(f"\nDữ liệu đã sẵn sàng tại: {TARGET_DATA_DIR}")
print(sorted(os.listdir(TARGET_DATA_DIR)))


## 5. Cell xác minh code đã có đúng patch

Cell này **cần viết riêng cho từng phương án** (không có hạ tầng chung, vì mỗi patch có tên
class/hàm/argument khác nhau). Mẫu tham khảo đầy đủ: xem Cell 5 trong
`phuong_an_1_OT_noise_scheduler/DiffMM_PhuongAn1_OT_Colab.ipynb` — kiểm tra sự tồn tại của
class/argument mới bằng cách đọc nội dung file .py rồi kiểm tra chuỗi đặc trưng, in ✓/✗ rõ ràng, và
`assert` toàn bộ điều kiện đều đúng trước khi cho phép chạy tiếp.

In [ ]:
# ============================================================
# CELL 5 — XÁC MINH REPO ĐÃ CLONE CÓ ĐÚNG PATCH PHƯƠNG ÁN 7
# ============================================================

params_path = os.path.join(REPO_DIR, "Params.py")
model_path = os.path.join(REPO_DIR, "Model.py")
main_path = os.path.join(REPO_DIR, "Main.py")

with open(params_path, "r", encoding="utf-8") as f:
    params_src = f.read()
with open(model_path, "r", encoding="utf-8") as f:
    model_src = f.read()
with open(main_path, "r", encoding="utf-8") as f:
    main_src = f.read()

checks = {
    "Params.py có --velocity_mode": "--velocity_mode" in params_src,
    "Model.py có class GaussianDiffusionTVS": "class GaussianDiffusionTVS" in model_src,
    "Model.py có class GaussianDiffusionAnchorOT (lop cha)": "class GaussianDiffusionAnchorOT" in model_src,
    "Main.py import GaussianDiffusionTVS": "GaussianDiffusionTVS" in main_src and "from Model import" in main_src,
    "Main.py khởi tạo bằng GaussianDiffusionTVS":
        "GaussianDiffusionTVS(args.sigma_min" in main_src,
}

for name, ok in checks.items():
    print(("✓ " if ok else "✗ ") + name)

assert all(checks.values()), (
    "Repo vừa clone KHÔNG có patch Phương án 7. Kiểm tra lại: bạn đã push đúng folder "
    "phuong_an_7_tvs/DiffMM-TVS chưa, và GITHUB_REPO_URL ở Cell 1 có trỏ đúng repo/branch đó không."
)
print("\n--- Repo đã clone đúng là bản có Phương án 7 (Triangle Velocities Synergy) ---")


## 6. Cell chạy huấn luyện

**Hạ tầng đã kiểm chứng — không cần sửa cấu trúc**, chỉ điền `DATASET_HP`/`CLI_ARGS` cho đúng lệnh
chạy của thuật toán mới (lấy từ README gốc). Chạy bằng `subprocess` (không dùng
`!cd ... && ... | tee ...`) để log có đường dẫn tuyệt đối, và để **báo lỗi ngay tại đây** (kèm exit
code + toàn bộ output) nếu script chính thoát với lỗi, thay vì để lỗi trôi xuống Cell 7 dưới dạng
`FileNotFoundError` khó hiểu.

In [ ]:
# ============================================================
# CELL 6 — CHẠY HUẤN LUYỆN
# ============================================================
import subprocess

DATASET_HP = {
    "tiktok": ["--reg", "1e-4", "--ssl_reg", "1e-2", "--trans", "1", "--e_loss", "0.1", "--cl_method", "1"],
    "baby":   ["--reg", "1e-5", "--ssl_reg", "1e-1", "--keepRate", "1", "--e_loss", "0.01"],
    "sports": ["--reg", "1e-6", "--ssl_reg", "1e-2", "--temp", "0.1", "--ris_lambda", "0.1", "--e_loss", "0.5", "--keepRate", "1", "--trans", "1"],
}

CLI_ARGS = [
    "--data", DATASET_NAME,
    "--epoch", str(NUM_EPOCHS),
    "--sigma_min", str(SIGMA_MIN),
    "--w_clip", str(W_CLIP),
    "--num_sample_steps", str(NUM_SAMPLE_STEPS),
    "--anchor_w", str(ANCHOR_W),
    "--velocity_mode", str(VELOCITY_MODE),
    "--lambda_x", str(LAMBDA_X),
    "--lambda_y", str(LAMBDA_Y),
    "--lambda_z", str(LAMBDA_Z),
    "--patience", str(PATIENCE),
] + DATASET_HP[DATASET_NAME]

LOG_PATH = os.path.abspath("train_log.txt")

cmd = ["python", MAIN_SCRIPT_NAME] + CLI_ARGS

print("Lệnh chạy:", " ".join(cmd))
print("Thư mục làm việc:", os.path.abspath(REPO_DIR))
print("Log sẽ được ghi vào:", LOG_PATH)
print()

with open(LOG_PATH, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, universal_newlines=True,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    process.wait()

print(f"\n\n{MAIN_SCRIPT_NAME} kết thúc với exit code: {process.returncode}")
assert process.returncode == 0, (
    f"{MAIN_SCRIPT_NAME} thoát với lỗi (exit code {process.returncode}). Xem log phía trên (hoặc mở "
    f"file {LOG_PATH}) để biết chi tiết lỗi — sửa xong thì chạy lại Cell 6 trước khi sang Cell 7."
)
assert os.path.exists(LOG_PATH) and os.path.getsize(LOG_PATH) > 0, f"Không tạo được file log tại {LOG_PATH}."
print(f"Huấn luyện xong, đã ghi log đầy đủ vào: {LOG_PATH}")


## 7. Cell xuất kết quả (bảng + file PDF tự động)

**Hạ tầng đã kiểm chứng (phần kiểm tra phòng vệ + phần xuất PDF) — chỉ cần điền `RESULT_REGEX` và
cách map kết quả**, y hệt trước đây. Đọc kỹ code phần in kết quả cuối cùng của script gốc (hoặc chạy
thử 1 lần) để biết chính xác định dạng dòng log chứa chỉ số tốt nhất, viết regex khớp đúng định dạng
đó — không đoán.

**Phần xuất PDF ở cuối cell không cần sửa gì** — tự động lấy tên phương án từ cột `"Phương án"` và tên
dataset từ biến `DATASET_NAME` (Cell 1) để đặt tên/tiêu đề file, dùng `matplotlib` (đã có sẵn trên
Colab, không cần cài thêm) để vẽ bảng `result_df` thành 1 trang PDF, rồi **tự động tải file PDF về máy
qua trình duyệt** (dùng `google.colab.files.download` — chỉ hoạt động khi chạy thật trên Colab).

In [ ]:
# ============================================================
# CELL 7 — XUẤT KẾT QUẢ (bảng + tự động xuất & tải file PDF)
# ============================================================
import re

import pandas as pd

assert "LOG_PATH" in globals() and os.path.exists(LOG_PATH), (
    "Không tìm thấy file log huấn luyện (biến LOG_PATH chưa có hoặc file không tồn tại). "
    "Nguyên nhân thường gặp nhất: Cell 6 chưa được chạy trong phiên này, hoặc phiên Colab đã bị "
    "reset/ngắt kết nối (mất hết file tạm) giữa lúc chạy Cell 6 và Cell 7. "
    "=> Hãy chạy lại Cell 6, đợi huấn luyện xong hẳn, rồi chạy lại Cell 7 này."
)

with open(LOG_PATH, "r", encoding="utf-8") as f:
    log_text = f.read()

RESULT_REGEX = r"Best epoch\s*:\s*(\d+)\s*,\s*Recall\s*:\s*([\d.]+)\s*,\s*NDCG\s*:\s*([\d.]+)\s*,\s*Precision\s*([\d.]+)"

m = re.search(RESULT_REGEX, log_text)
assert m, (
    f"Cell 6 đã chạy xong (log tồn tại tại {LOG_PATH}) nhưng không tìm thấy kết quả khớp RESULT_REGEX "
    "trong log — kiểm tra lại regex đã đúng định dạng dòng in kết quả thật của script gốc chưa, hoặc "
    "mở log lên xem script có báo lỗi gì trước đó không."
)

best_epoch, recall, ndcg, precision = m.groups()

# Trích xuất toàn bộ chỉ số qua các epoch
epochs = []
recalls = []
ndcgs = []
precisions = []

pattern = r"Epoch\s+(\d+)/\d+,\s+Test:\s+Recall\s*=\s*([\d.]+),\s+NDCG\s*=\s*([\d.]+),\s+Precision\s*=\s*([\d.]+)"
for line in log_text.split('\n'):
    m_line = re.search(pattern, line)
    if m_line:
        ep_num, r_val, n_val, p_val = m_line.groups()
        epochs.append(int(ep_num))
        recalls.append(float(r_val))
        ndcgs.append(float(n_val))
        precisions.append(float(p_val))

all_epochs_df = pd.DataFrame({
    "Epoch": epochs,
    "Recall@20": [f"{v:.5f}" for v in recalls],
    "NDCG@20": [f"{v:.5f}" for v in ndcgs],
    "Precision@20": [f"{v:.5f}" for v in precisions]
})

print("=== BẢNG CHỈ SỐ CHI TIẾT QUA TẤT CẢ EPOCHS HUẤN LUYỆN ===")
display(all_epochs_df)
print()
print(all_epochs_df.to_markdown(index=False, floatfmt=".5f"))
print('\n' + '='*80 + '\n')

result_df = pd.DataFrame([{
    "Dataset": DATASET_NAME,
    "Phương án": "Phương án 7 (TVS)",
    "Best Epoch": int(best_epoch),
    "Recall@20": f"{float(recall):.5f}",
    "NDCG@20": f"{float(ndcg):.5f}",
    "Precision@20": f"{float(precision):.5f}",
    "VELOCITY_MODE": VELOCITY_MODE,
    "ANCHOR_W": ANCHOR_W,
    "SIGMA_MIN": SIGMA_MIN,
    "W_CLIP": W_CLIP,
    "SAMPLE_STEPS": NUM_SAMPLE_STEPS,
    "LAMBDAS": f"({LAMBDA_X},{LAMBDA_Y},{LAMBDA_Z})",
}])

print("=== BẢNG CHỈ SỐ TỐT NHẤT (BEST EPOCH) ===")
display(result_df)
print()
print(result_df.to_markdown(index=False, floatfmt=".5f"))

# ------------------------------------------------------------------
# Xuất bảng kết quả cao nhất + tên dataset ra file PDF, rồi tự động tải về máy — KHÔNG cần sửa gì
# bên dưới, phần này dùng chung cho mọi phương án (tự lấy dữ liệu từ result_df/DATASET_NAME ở trên).
# ------------------------------------------------------------------
import matplotlib.pyplot as plt
from datetime import datetime


def _export_result_pdf(df, title_lines, out_path):
    n_rows, n_cols = len(df), len(df.columns)
    fig_w = min(max(6, 1.6 * n_cols), 18)
    fig_h = 1.8 + 0.55 * (n_rows + 1)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    ax.set_title("\n".join(title_lines), fontsize=11, fontweight="bold", loc="left", pad=14)
    tbl = ax.table(
        cellText=df.astype(str).values,
        colLabels=df.columns,
        cellLoc="center",
        loc="upper center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.auto_set_column_width(col=list(range(n_cols)))
    tbl.scale(1, 1.7)
    fig.tight_layout()
    fig.savefig(out_path, format="pdf", bbox_inches="tight")
    plt.close(fig)


_phuong_an_name = result_df["Phương án"].iloc[0] if "Phương án" in result_df.columns else "DiffMM"
_dataset_name = DATASET_NAME if "DATASET_NAME" in globals() and DATASET_NAME else (
    result_df["Dataset"].iloc[0] if "Dataset" in result_df.columns else "unknown_dataset"
)
_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

PDF_PATH = os.path.abspath(f"KetQua_{_dataset_name}.pdf")
_export_result_pdf(
    result_df,
    title_lines=[
        f"Kết quả huấn luyện — {_phuong_an_name}",
        f"Dataset: {_dataset_name}    |    Xuất lúc: {_timestamp}",
        f"Configs: Epochs={NUM_EPOCHS}, VelMode={VELOCITY_MODE}, AnchorW={ANCHOR_W}, SigmaMin={SIGMA_MIN}, WClip={W_CLIP}, SampleSteps={NUM_SAMPLE_STEPS}, Lambdas=({LAMBDA_X},{LAMBDA_Y},{LAMBDA_Z})",
    ],
    out_path=PDF_PATH,
)
print(f"\nĐã xuất file PDF kết quả: {PDF_PATH}")

try:
    from google.colab import files as _colab_files
    _colab_files.download(PDF_PATH)
    print("Đã tự động tải file PDF về máy (kiểm tra thư mục Downloads của trình duyệt).")
except ImportError:
    print("Không chạy trong Colab nên không tự tải xuống — file PDF vẫn đã được lưu ở đường dẫn trên.")

if epochs:
    import matplotlib.ticker as ticker
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    
    # Plot Recall
    axes[0].plot(epochs, recalls, marker='o', color='#3b82f6', linewidth=2, label='Recall@20')
    axes[0].set_title('Recall@20 over Epochs', fontsize=12, fontweight='bold', pad=10)
    axes[0].set_xlabel('Epoch', fontsize=10)
    axes[0].set_ylabel('Recall', fontsize=10)
    axes[0].set_xlim(0, NUM_EPOCHS)
    axes[0].set_xticks(range(0, NUM_EPOCHS + 1, max(1, NUM_EPOCHS // 10)))
    axes[0].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.5f'))
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(loc='lower right')
    
    # Plot NDCG
    axes[1].plot(epochs, ndcgs, marker='s', color='#f59e0b', linewidth=2, label='NDCG@20')
    axes[1].set_title('NDCG@20 over Epochs', fontsize=12, fontweight='bold', pad=10)
    axes[1].set_xlabel('Epoch', fontsize=10)
    axes[1].set_ylabel('NDCG', fontsize=10)
    axes[1].set_xlim(0, NUM_EPOCHS)
    axes[1].set_xticks(range(0, NUM_EPOCHS + 1, max(1, NUM_EPOCHS // 10)))
    axes[1].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.5f'))
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(loc='lower right')
    
    # Plot Precision
    axes[2].plot(epochs, precisions, marker='^', color='#10b981', linewidth=2, label='Precision@20')
    axes[2].set_title('Precision@20 over Epochs', fontsize=12, fontweight='bold', pad=10)
    axes[2].set_xlabel('Epoch', fontsize=10)
    axes[2].set_ylabel('Precision', fontsize=10)
    axes[2].set_xlim(0, NUM_EPOCHS)
    axes[2].set_xticks(range(0, NUM_EPOCHS + 1, max(1, NUM_EPOCHS // 10)))
    axes[2].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.5f'))
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].legend(loc='lower right')
    
    fig.suptitle(f'Biến thiên chỉ số huấn luyện — {DATASET_NAME.upper()} ({_phuong_an_name})', fontsize=14, fontweight='bold', y=1.05)
    fig.tight_layout()
    
    PLOT_PNG_PATH = os.path.abspath(f"BieuDo_{_dataset_name}.png")
    PLOT_PDF_PATH = os.path.abspath(f"BieuDo_{_dataset_name}.pdf")
    fig.savefig(PLOT_PNG_PATH, format="png", dpi=150, bbox_inches="tight")
    fig.savefig(PLOT_PDF_PATH, format="pdf", bbox_inches="tight")
    plt.show()
    print(f"Đã xuất biểu đồ: {PLOT_PNG_PATH} và {PLOT_PDF_PATH}")
    
    try:
        from google.colab import files as _colab_files
        _colab_files.download(PLOT_PDF_PATH)
        print("Đã tự động tải file PDF biểu đồ về máy.")
    except ImportError:
        pass
else:
    print("Không tìm thấy dữ liệu Test trong log để vẽ biểu đồ.")


In [ ]:
# ============================================================
# CELL 8 (TÙY CHỌN) — TỐI ƯU HÓA SIÊU THAM SỐ BẰNG OPTUNA
# ============================================================
# Chạy cell này nếu bạn muốn tự động tìm bộ tham số tốt nhất cho TVS.
# ------------------------------------------------------------
import os
import re
import subprocess

try:
    import optuna
except ImportError:
    print("Đang cài đặt Optuna...")
    subprocess.run(["pip", "install", "-q", "optuna"])
    import optuna

N_TRIALS = 10  # Số lượt thử nghiệm. Hãy tăng lên 20-30 để tìm kết quả tốt hơn.
OPTUNA_EPOCHS = 25  # Số epoch chạy thử mỗi lượt (đảm bảo bao phủ đỉnh hội tụ từ 15-25)

def objective(trial):
    # 1. Gợi ý các tham số cần tối ưu
    opt_anchor_w = trial.suggest_float("anchor_w", 0.5, 5.0)
    opt_lambda_y = trial.suggest_float("lambda_y", 0.1, 1.0)
    opt_lambda_z = trial.suggest_float("lambda_z", 0.1, 1.0)
    opt_sigma_min = trial.suggest_float("sigma_min", 1e-4, 5e-3, log=True)
    
    print(f"\n[Trial {trial.number}] Đang chạy thử nghiệm với: anchor_w={opt_anchor_w:.4f}, lambda_y={opt_lambda_y:.4f}, lambda_z={opt_lambda_z:.4f}, sigma_min={opt_sigma_min:.6f}")
    
    # 2. Xóa log cũ trước khi chạy
    LOG_PATH = os.path.abspath("train_log.txt")
    if os.path.exists(LOG_PATH):
        os.remove(LOG_PATH)
        
    # 3. Chuẩn bị tham số chạy
    DATASET_HP = {
        "tiktok": ["--reg", "1e-4", "--ssl_reg", "1e-2", "--trans", "1", "--e_loss", "0.1", "--cl_method", "1"],
        "baby":   ["--reg", "1e-5", "--ssl_reg", "1e-1", "--keepRate", "1", "--e_loss", "0.01"],
        "sports": ["--reg", "1e-6", "--ssl_reg", "1e-2", "--temp", "0.1", "--ris_lambda", "0.1", "--e_loss", "0.5", "--keepRate", "1", "--trans", "1"],
    }
    
    CLI_ARGS = [
        "--data", DATASET_NAME,
        "--epoch", str(OPTUNA_EPOCHS),
        "--sigma_min", str(opt_sigma_min),
        "--w_clip", str(W_CLIP),
        "--num_sample_steps", str(NUM_SAMPLE_STEPS),
        "--anchor_w", str(opt_anchor_w),
        "--velocity_mode", "1",
        "--lambda_x", "1.0",
        "--lambda_y", str(opt_lambda_y),
        "--lambda_z", str(opt_lambda_z),
        "--patience", "0",
    ] + DATASET_HP[DATASET_NAME]
    
    cmd = ["python", MAIN_SCRIPT_NAME] + CLI_ARGS
    
    # 4. Chạy mô hình
    with open(LOG_PATH, "w", encoding="utf-8") as log_file:
        process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=log_file, stderr=subprocess.STDOUT)
        process.wait()
        
    # 5. Đọc kết quả tốt nhất từ log
    if not os.path.exists(LOG_PATH):
        return 0.0
        
    with open(LOG_PATH, "r", encoding="utf-8") as f:
        log_text = f.read()
        
    RESULT_REGEX = r"Best epoch\s*:\s*\d+\s*,\s*Recall\s*:\s*([\d.]+)"
    m = re.search(RESULT_REGEX, log_text)
    if m:
        best_recall = float(m.group(1))
        print(f"-> Lượt chạy kết thúc. Recall@20 tốt nhất đạt: {best_recall:.5f}")
        return best_recall
    else:
        print("-> Lượt chạy thất bại hoặc không ghi nhận kết quả. Đánh giá: 0.0")
        return 0.0

print("Bắt đầu tối ưu hóa bằng Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print("\n" + "="*40)
print("TỐI ƯU HÓA HOÀN TẤT!")
print("Bộ tham số tốt nhất:", study.best_params)
print(f"Recall@20 tốt nhất đạt: {study.best_value:.5f}")
